# Train YOLO11n on Nebius Serverless AI Jobs — from a notebook

Presenter notebook, agenda 75–95 min. Old school: export → stage → submit → poll → pull weights → predict → evaluate. No plugin, no panel.

Prereqs on this laptop: `nebius` CLI logged in (`nebius profile create`), the Object Storage access key at `~/Documents/api/nebius/workshop-s3-key.json`, and this repo's venv (`pip install nebius boto3 ultralytics`).

**Timing on stage:** upload ≈ 1 min, provisioning ≈ 2–3 min, 15 epochs on an L40S ≈ 3–4 min. Start the submit cell *before* the auto-label discussion ends. If the job is late, `presenter/weights/best.pt` from the dry run is the fallback.

In [ ]:
import os, sys, time
sys.path.insert(0, os.path.abspath(".."))

import fiftyone as fo
from fiftyone import ViewField as F
from presenter.nebius_job import NebiusJob

RUN = "stuttgart"                       # munich / berlin: one run per stop, or reuse
frames = fo.load_dataset("droid-frames")
print(len(frames), "frames;", frames.count_values("split"))

## 1. Export the curated, auto-labeled frames to YOLO format

Split is by episode. Near-duplicates excluded from train. (`build/export_yolo.py` does the same from the CLI.)

In [ ]:
!python ../build/export_yolo.py --out ../build/yolo_export
!cat ../build/yolo_export/dataset.yaml

## 2. Stage on Nebius Object Storage

S3-compatible bucket `physical-ai-workshop` in `eu-north1`. The GPU job pulls from here and writes `best.pt` back.

In [ ]:
job = NebiusJob()
input_uri = job.upload_dir("../build/yolo_export", f"runs/{RUN}/input")
print(input_uri)

## 3. Submit the job

One L40S. The container is the public `ghcr.io/danielgural/fiftyone-yolo-train` image: it syncs `INPUT_S3_URI` to `/data`, runs `YOLO(MODEL).train(data=/data/dataset.yaml, **HYPERPARAMS_JSON)`, and uploads `best.pt` to `OUTPUT_S3_URI`.

In [ ]:
job_id = job.submit(RUN, f"runs/{RUN}", model="yolo11n.pt", epochs=15, imgsz=640, batch=32,
                    platform="gpu-l40s-a", preset="1gpu-8vcpu-32gb", timeout_h=1)
job_id

Show the job in the Nebius console while it provisions: **console.nebius.com → AI Jobs**. Then poll:

In [ ]:
state = job.wait(job_id, every=15)     # PROVISIONING → IMAGE_PULLING → RUNNING → COMPLETED
state

In [ ]:
print(job.logs(job_id)[-3000:])         # last lines of the training log

## 4. Pull the weights and predict

In [ ]:
best = job.download(f"runs/{RUN}/output/best.pt", f"weights/best-{RUN}.pt")
best

In [ ]:
from ultralytics import YOLO
model = YOLO(best)
val = frames.match(F("split") == "val")
val.apply_model(model, label_field="yolo11n_preds", confidence_thresh=0.25)
print(val.count_values("yolo11n_preds.detections.label"))

## 5. Evaluate against the auto-labels

Open the **Model Evaluation** panel on `eval_yolo` in the App; the confusion-matrix cells are clickable.

In [ ]:
if "eval_yolo" in frames.list_evaluations():
    frames.delete_evaluation("eval_yolo")
results = val.evaluate_detections("yolo11n_preds", gt_field="auto_labels", eval_key="eval_yolo", compute_mAP=True)
results.print_report()
print("mAP", round(results.mAP(), 3))

In [ ]:
session = fo.launch_app(frames)
session.view = val.sort_by("eval_yolo_fp", reverse=True)

## 6. Back to the recording

`../notebooks/06_close_the_loop.ipynb` turns `yolo11n_preds` into `brick-visible` / `gripper-visible` temporal tags on the MCAP episodes. For the deployment, the same `apply_model` runs on the `Droid frames` dataset in demo.fiftyone.ai (see `RUNBOOK.md`).